In [ ]:
import pandas as pd
import numpy as np

# Correct path pointing to the processed EDA file
df_cleaned = pd.read_csv('../data/processed/berlin_airbnb_cleaned_EDA.csv') 

# Check the dimensions (how many rows and columns it has now)
print(f"The cleaned dataset has {df_cleaned.shape[0]} rows and {df_cleaned.shape[1]} columns.")

# Display the first few rows to inspect the data structure
df_cleaned.head()

# Check data types and missing values for each column
df_cleaned.info()

In [ ]:
# List of columns to drop because they do not provide text/numerical value for price prediction
columns_to_drop = [
    'index', 'Review ID', 'review_date', 'Reviewer ID', 'Reviewer Name', 
    'Comments', 'Listing ID', 'Listing URL', 'Listing Name', 'Host ID', 
    'Host URL', 'Host Name', 'Host Since', 'City', 'Postal Code'
]

# Drop the specified columns and create a new DataFrame dedicated to Machine Learning
df_ml = df_cleaned.drop(columns=columns_to_drop, errors='ignore')

# Verify the new dimensions and see which columns are left
print(f"The new dataset for Machine Learning has {df_ml.shape[0]} rows and {df_ml.shape[1]} columns.")
print("\nRemaining columns:")
print(df_ml.columns.tolist())

In [ ]:
# Check the main descriptive statistics for the price column
print("Price column descriptive statistics:")
print(df_ml['Price'].describe())

# Check the 5 highest prices to spot extreme outliers in the dataset
print("\nTop 5 highest prices:")
print(df_ml['Price'].nlargest(5))

# Check if there are any invalid, zero, or negative prices
print("\nNumber of rows where price is 0 or less:")
print((df_ml['Price'] <= 0).sum())

In [ ]:
# Count how many listings have a price above 300 Euro
expensive_listings = (df_ml['Price'] > 300).sum()
print(f"Number of listings with a price above 300: {expensive_listings}")

# Filter the dataset to keep only realistic prices (less than or equal to 300 Euro)
df_ml = df_ml[df_ml['Price'] <= 300]

# Verify the new dataset dimensions and the new maximum price after outlier removal
print(f"\nNew dataset shape after removing outliers: {df_ml.shape}")
print(f"New maximum price: {df_ml['Price'].max()}")

In [ ]:
# Check which of the remaining columns still contain missing values (NaN)
missing_counts = df_ml.isnull().sum()
missing_columns = missing_counts[missing_counts > 0].sort_values(ascending=False)

print("Remaining columns with missing values and their counts:")
if not missing_columns.empty:
    print(missing_columns)
else:
    print("Perfect! No missing values found in the remaining columns.")

In [ ]:
# 1. Drop the date columns as they are not directly useful for ML algorithms
df_ml = df_ml.drop(columns=['First Review', 'Last Review'], errors='ignore')

# 2. Fill missing categorical/text values with standard placeholders
df_ml['Host Response Time'] = df_ml['Host Response Time'].fillna('Unknown')
df_ml['Is Superhost'] = df_ml['Is Superhost'].fillna('False')

# List of numerical columns that contain missing values and need imputation
numerical_missing = [
    'Host Response Rate', 'Cleanliness Rating', 'Location Rating', 
    'Accuracy Rating', 'Communication Rating', 'Checkin Rating', 'Value Rating'
]

# 3. Clean characters, force convert columns to numeric, and impute missing gaps using the median
for col in numerical_missing:
    # Remove the percentage sign (%) by converting to string first, just in case
    df_ml[col] = df_ml[col].astype(str).str.replace('%', '', regex=False)
    
    # Force convert to numeric format (any invalid text like 'nan' automatically becomes NaN)
    df_ml[col] = pd.to_numeric(df_ml[col], errors='coerce')
    
    # Calculate the median safely and fill the remaining missing gaps
    median_value = df_ml[col].median()
    df_ml[col] = df_ml[col].fillna(median_value)

# 4. Final verification check to ensure no missing values remain
print("Remaining missing values count:")
print(df_ml.isnull().sum().sum())

In [ ]:
# =====================================================================
# 🛠️ ONE-HOT ENCODING & FINAL SAVE (CLEANED VERSION)
# =====================================================================

# 1. Select all categorical columns that are still text (object/string)
categorical_cols = df_ml.select_dtypes(include=['object', 'str']).columns.tolist()
print(f"Categorical columns to encode: {categorical_cols}")

# 2. Apply One-Hot Encoding using pandas get_dummies
df_final_ml = pd.get_dummies(df_ml, columns=categorical_cols, drop_first=True, dtype=int)

# Print the final shape of our machine learning dataset
print(f"\nFinal dataset shape for Machine Learning: {df_final_ml.shape}")

# 3. Save the final prepared dataset to the correct processed folder
# Kujdes: Këtu do të vendosim emrin e ri të skedarit që të jetë korrekt!
output_path = '../data/processed/berlin_airbnb_cleaned_ML.csv'
df_final_ml.to_csv(output_path, index=False)

print(f"\n[SUCCESS] The clean dataset has been saved to: {output_path}")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset to recover the original text columns (like neighbourhood names)
try:
    df_raw = pd.read_csv('../data/raw/berlin_airbnb.csv')
except:
    df_raw = pd.read_csv('../data/berlin_airbnb.csv')

# 2. Load your current ML dataset to use as a master filter for rows
df_ml = pd.read_csv('../data/processed/berlin_airbnb_cleaned_ML.csv')

# 3. Drop 'Square Feet' and other completely messy columns from the raw data immediately
bad_cols = ['Square Feet', 'square_feet', 'id', 'listing_url', 'scrape_id', 'last_scraped', 'name']
df_raw_filtered = df_raw.drop(columns=[c for c in bad_cols if c in df_raw.columns], errors='ignore')

# 4. Keep ONLY the rows that match your final cleaned ML dataset
# We use the index to align them perfectly
df_eda_perfect = df_raw_filtered.iloc[df_ml.index].copy()

# 5. Quick clean of the core numericals in this perfectly sliced dataset
if 'Price' in df_eda_perfect.columns and df_eda_perfect['Price'].dtype == 'object':
    df_eda_perfect['Price'] = df_eda_perfect['Price'].str.replace('$', '').str.replace(',', '').astype(float)
elif 'price' in df_eda_perfect.columns and df_eda_perfect['price'].dtype == 'object':
    df_eda_perfect['price'] = df_eda_perfect['price'].str.replace('$', '').str.replace(',', '').astype(float)

# Fill remaining core missing fields with median/mode based on the filtered rows
for col in df_eda_perfect.columns:
    if df_eda_perfect[col].dtype in ['int64', 'float64']:
        df_eda_perfect[col] = df_eda_perfect[col].fillna(df_eda_perfect[col].median())

# 6. Save both datasets separately in the processed data folder
# Save the 46-column version for EDA and visualizations
df_eda_perfect.to_csv('../data/processed/berlin_airbnb_cleaned_EDA.csv', index=False)

# Save the final encoded version for Machine Learning models
df_ml.to_csv('../data/processed/berlin_airbnb_cleaned_ML.csv', index=False)

print(f"🎯 Success! Datasets synchronized and saved perfectly.")
print(f"Rows in ML: {df_ml.shape[0]}  |  Rows in EDA: {df_eda_perfect.shape[0]}")